# Notebook 6: Results Analysis and Thesis Tables

**Goal**: Compile ALL results from notebooks 2-5 into:
1. The master comparison table (for your thesis)
2. Publication-quality plots
3. Statistical analysis (error reduction, relative improvement)
4. Discussion points for your thesis

**Run this LAST, after all experiments are complete.**

No GPU required for this notebook.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from google.colab import drive

drive.mount('/content/drive')
WORKSPACE = '/content/drive/MyDrive/thesis_mbert'

sns.set_theme(style='whitegrid', font_scale=1.2)
print('Libraries loaded.')

In [ ]:
# ============================================================
# Paper Results (ground truth for comparison)
# ============================================================

PAPER = {
    'FASTT':       {'ga': 65.36, 'mt': 68.23, 'sing': 66.42, 'vi': 53.37},
    'ELMo':        {'ga': 68.25, 'mt': 74.33, 'sing': 68.63, 'vi': 56.91},
    'MBERT':       {'ga': 68.19, 'mt': 67.06, 'sing': 74.01, 'vi': 62.96},
    'LAPT':        {'ga': 73.03, 'mt': 78.51, 'sing': 76.48, 'vi': 64.67},
    'VA':          {'ga': 72.68, 'mt': 79.88, 'sing': 76.71, 'vi': 64.28},
    'TVA':         {'ga': 72.95, 'mt': 79.32, 'sing': 76.88, 'vi': 64.40},
    'MBERT+FT':    {'ga': 72.67, 'mt': 76.74, 'sing': 78.24, 'vi': 66.13},
    'LAPT+FT':     {'ga': 75.45, 'mt': 82.77, 'sing': 79.30, 'vi': 67.50},
    'VA+FT':       {'ga': 76.17, 'mt': 83.53, 'sing': 79.89, 'vi': 67.28},
    'TVA+FT':      {'ga': 75.95, 'mt': 83.10, 'sing': 80.10, 'vi': 67.35},
}

LANGS = ['ga', 'mt', 'sing', 'vi']
LANG_NAMES = {'ga': 'Irish (T1)', 'mt': 'Maltese (T2)', 'sing': 'Singlish (T0)', 'vi': 'Vietnamese (T0)'}

print('Paper results loaded.')

In [ ]:
# ============================================================
# Load our experimental results
# ============================================================

def load_result(experiment_name, lang):
    """Load a saved result from a specific experiment."""
    path = os.path.join(WORKSPACE, 'results', experiment_name, lang, 'result.json')
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

def get_las(experiment_name, lang, default=None):
    r = load_result(experiment_name, lang)
    return r['test_las'] if r else default

# Define all our experiments: (display name, experiment directory name)
OUR_EXPERIMENTS = [
    ('MBERT frozen (repro)',    'mbert_frozen'),
    ('MBERT+FT (repro)',        'mbert_ft'),
    ('LAPT+FT (repro)',         'lapt_ft'),
    ('VA+FT (repro)',           'va_ft'),
    # Innovation 1: XLM-R
    ('XLM-R frozen',            'xlmr_frozen'),
    ('XLM-R+FT',                'xlmr_ft'),
    ('XLM-R+LAPT+FT',           'xlmr_lapt_ft'),
    # Innovation 2: LoRA
    ('LoRA-LAPT+FT',            'lora_lapt_ft'),
]

# Build a DataFrame with all results
rows = []

# Paper results first
for method, vals in PAPER.items():
    row = {'Method': method, 'Source': 'Paper'}
    for lang in LANGS:
        row[LANG_NAMES[lang]] = vals.get(lang, None)
    rows.append(row)

# Our results
for display_name, exp_name in OUR_EXPERIMENTS:
    row = {'Method': display_name, 'Source': 'Ours'}
    for lang in LANGS:
        row[LANG_NAMES[lang]] = get_las(exp_name, lang)
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# ============================================================
# Main Results Table (LaTeX format for your thesis)
# ============================================================

print('LaTeX table for thesis:')
print('\\begin{table*}[t]')
print('\\centering')
print('\\begin{tabular}{lcccc}')
print('\\toprule')
print(f'Method & Irish (T1) & Maltese (T2) & Singlish (T0) & Vietnamese (T0) \\\\')
print('\\midrule')
print('\\textit{Paper baselines} \\\\')

paper_methods = ['MBERT', 'LAPT+FT', 'VA+FT']
for method in paper_methods:
    vals = [PAPER[method].get(l, '-') for l in LANGS]
    row = f'{method} & ' + ' & '.join([f'{v:.2f}' if isinstance(v, float) else str(v) for v in vals]) + ' \\\\'
    print(row)

print('\\midrule')
print('\\textit{Our methods} \\\\')

our_methods = [
    ('XLM-R+FT', 'xlmr_ft'),
    ('XLM-R+LAPT+FT', 'xlmr_lapt_ft'),
    ('LoRA-LAPT+FT', 'lora_lapt_ft'),
]
for display, exp_name in our_methods:
    vals = [get_las(exp_name, l) for l in LANGS]
    formatted = [f'{v:.2f}' if v else '--' for v in vals]
    row = f'{display} & ' + ' & '.join(formatted) + ' \\\\'
    print(row)

print('\\bottomrule')
print('\\end{tabular}')
print('\\caption{LAS scores on UD 2.5 dependency parsing. T0/T1/T2 refers to the')
print('language type with respect to mBERT pretraining data.}')
print('\\label{tab:main_results}')
print('\\end{table*}')

In [ ]:
# ============================================================
# Error Reduction Analysis
# ============================================================
# The paper reports error reduction from MBERT to each method.
# Error reduction = (method LAS - mbert LAS) / (100 - mbert LAS) * 100

print('\nRelative Error Reduction from MBERT+FT baseline:')
print(f'{"Method":<25}', *[f'{LANG_NAMES[l]:>15}' for l in LANGS])
print('-' * 90)

mbert_ft_scores = PAPER['MBERT+FT']

comparisons = [
    ('LAPT+FT (paper)',   PAPER['LAPT+FT']),
    ('VA+FT (paper)',     PAPER['VA+FT']),
]

# Add our innovations
for display, exp_name in [('XLM-R+FT', 'xlmr_ft'), ('XLM-R+LAPT+FT', 'xlmr_lapt_ft'), ('LoRA-LAPT+FT', 'lora_lapt_ft')]:
    vals = {l: get_las(exp_name, l) for l in LANGS}
    if any(v for v in vals.values()):
        comparisons.append((display, vals))

for method_name, scores in comparisons:
    row = []
    for lang in LANGS:
        method_las = scores.get(lang)
        baseline   = mbert_ft_scores.get(lang, 100)
        if method_las and baseline:
            reduction = (method_las - baseline) / (100 - baseline) * 100
            row.append(f'{reduction:>14.1f}%')
        else:
            row.append(f'{"N/A":>15}')
    print(f'{method_name:<25}', *row)

print('\nNote: Error reduction = (method - baseline) / (100 - baseline) * 100')
print('Positive = improvement over mBERT+FT.')

In [ ]:
# ============================================================
# Main comparison bar chart
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(18, 6), sharey=False)

methods_to_plot = [
    ('MBERT+FT\n(paper)', PAPER['MBERT+FT'], '#4472C4'),
    ('LAPT+FT\n(paper)',  PAPER['LAPT+FT'],  '#ED7D31'),
    ('VA+FT\n(paper)',    PAPER['VA+FT'],    '#FFC000'),
    ('XLM-R+FT\n(ours)', {l: get_las('xlmr_ft', l) or 0 for l in LANGS}, '#70AD47'),
    ('XLM-R+LAPT+FT\n(ours)', {l: get_las('xlmr_lapt_ft', l) or 0 for l in LANGS}, '#5A9E6F'),
    ('LoRA-LAPT+FT\n(ours)', {l: get_las('lora_lapt_ft', l) or 0 for l in LANGS}, '#9E5A9E'),
]

for ax, lang in zip(axes, LANGS):
    labels = [m[0] for m in methods_to_plot]
    values = [m[1].get(lang, 0) for m in methods_to_plot]
    colors = [m[2] for m in methods_to_plot]

    bars = ax.bar(range(len(labels)), values, color=colors, edgecolor='white', linewidth=0.5)

    # Add value labels on bars
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=8, rotation=45)

    ax.set_title(LANG_NAMES[lang], fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('LAS Score' if lang == 'ga' else '')
    ax.set_ylim(55, 92)
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Dependency Parsing Results: Paper Replication + Innovations',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

plot_path = os.path.join(WORKSPACE, 'results', 'main_results.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

In [ ]:
# ============================================================
# Training Curves (from saved logs)
# ============================================================

def load_training_log(experiment_name, lang):
    path = os.path.join(WORKSPACE, 'results', experiment_name, lang, 'training_log.json')
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

LANG_PLOT = 'mt'  # Maltese shows the most interesting curves

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

experiments_to_plot = [
    ('MBERT+FT', 'mbert_ft', '#4472C4'),
    ('LAPT+FT', 'lapt_ft', '#ED7D31'),
    ('XLM-R+FT', 'xlmr_ft', '#70AD47'),
    ('LoRA-LAPT+FT', 'lora_lapt_ft', '#9E5A9E'),
]

for ax, (metric, title) in zip(axes, [('dev_las', 'Validation LAS'), ('train_loss', 'Training Loss')]):
    for label, exp_name, color in experiments_to_plot:
        log = load_training_log(exp_name, LANG_PLOT)
        if log:
            epochs = [e['epoch'] for e in log]
            values = [e[metric] for e in log]
            ax.plot(epochs, values, label=label, color=color, linewidth=2)

    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.set_title(f'{title} — Maltese (Type 2)')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.4)

plt.tight_layout()
curve_path = os.path.join(WORKSPACE, 'results', 'training_curves_mt.png')
plt.savefig(curve_path, dpi=150)
plt.show()
print(f'Saved: {curve_path}')

In [ ]:
# ============================================================
# Thesis Discussion Points (auto-generated)
# ============================================================

print('=' * 70)
print('THESIS DISCUSSION POINTS')
print('=' * 70)

mbert_ft = PAPER['MBERT+FT']
lapt_ft  = PAPER['LAPT+FT']

print('\n1. Effectiveness of LAPT (from paper):')
for lang in LANGS:
    improvement = lapt_ft[lang] - mbert_ft[lang]
    print(f'   {LANG_NAMES[lang]}: +{improvement:.2f} LAS points')

print('\n2. XLM-R vs mBERT (our contribution):')
for lang in LANGS:
    xlmr_las = get_las('xlmr_ft', lang)
    if xlmr_las:
        diff = xlmr_las - mbert_ft[lang]
        print(f'   {LANG_NAMES[lang]}: XLM-R+FT {xlmr_las:.2f} vs mBERT+FT {mbert_ft[lang]:.2f} (diff={diff:+.2f})')

print('\n3. LoRA-LAPT vs Full LAPT (our contribution):')
for lang in LANGS:
    lora_las = get_las('lora_lapt_ft', lang)
    if lora_las:
        diff = lora_las - lapt_ft[lang]
        print(f'   {LANG_NAMES[lang]}: LoRA-LAPT {lora_las:.2f} vs Full-LAPT {lapt_ft[lang]:.2f} (diff={diff:+.2f})')

print('\n4. Language type correlation (key finding from paper):')
print('   Type 2 (Maltese, unseen in mBERT) benefits most from LAPT.')
print('   Type 1 (Irish, low-resource) benefits significantly.')
print('   Type 0 (Singlish, Vietnamese) benefit less.')
print('   CHECK: Does this pattern hold for XLM-R and LoRA-LAPT?')
print()
print('These findings form the core of your thesis Chapter 4 (Results and Analysis).')